# Goal:

So far, we have always been using a least squares cost function defined as

$$f(A, K, B) := \sum_{j \in \text{POVM}} \sum_{i \in \text{Sequences}} \left(p_{i,j}(A, K, B) - y_{i,j} \right)^2 $$


Instead, a well theoretically-motivated cost function is the log-likelihood function

$$\log L(A, K, B) := \sum_{j \in \text{POVM}} \sum_{i \in \text{Sequences}} y_{i,j} \log [p_{i,j}(A, K, B)] $$

The goal of this notebook is to

0. Implement the log-likelihood function. ✅
1. Test the values agree with roughly the square root of the least squares and the ones from MGST ✅
2. Test gradient computes and time against Least squares 
2. Perform optimization with it.

In [20]:
import jax.numpy as jnp

from iqm.benchmarks.compressive_gst.compressive_gst import GSTConfiguration
from mGST.utility_functions_comparisons import all_operators_from_configuration, kraus_tensor_to_mgst, get_compressed_rep_from_mgst_output, get_compressed_perturbed_kraus_from_superop, get_compressed_perturbed_rep_from_mgst, get_mgst_tensors_from_psd_representation
from mGST.simulate import generate_sequence_indices, compute_probability_matrices
from mGST.low_level_jit import cost_function_jax_mps, objf
from mGST.automatic_diff import automatic_gradient

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Get the tensors to test the cost function with

In [2]:
shots = 1000
num_sequences = 100
backend = "iqmfakeapollo" # just to create the benchmark and then get the gate set

In [3]:
Minimal_1Q_GST = GSTConfiguration(
    qubit_layouts=[[0]],
    gate_set="1QXYI",
    num_circuits=num_sequences,
    shots=shots,
    rank=4,
    verbose_level=2,
    convergence_criteria =  [3, 1e-20],
    max_iterations = [0,40],
    opt_method = "SFN",
    from_init = True,
    )

target_operators = all_operators_from_configuration(Minimal_1Q_GST, backend)
print(target_operators.keys())

# now get the superoperator for Kraus
povm_vect_target = target_operators["povm"]
state_vect_target = target_operators["state"]
kraus_tensor_unitary_target = target_operators["kraus"]
kraus_superop_target = kraus_tensor_to_mgst(kraus_tensor_unitary_target)

INFO:2026-07-16 16:03:04,601:jax._src.xla_bridge:927: Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
2026-07-16 16:03:04,601 - jax._src.xla_bridge - INFO - Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:2026-07-16 16:03:04,605:jax._src.xla_bridge:927: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/emiliano.godinez/.pyenv/versions/3.11.10/lib/libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Users/emiliano.godinez/.pyenv/versions/3.11.10/lib/libtpu.so' (no such file), '/opt/homebrew/lib/libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/lib/libtpu.so' (no such file), '/Users/emiliano.godinez/.pyenv/versions/3.11.10/lib/libtpu.so' (no such file), '/System/Volumes/Preboot/

dict_keys(['kraus', 'povm', 'state', 'gate_labels'])


In [4]:
num_qubits = 1
dim = 2**num_qubits
kraus_rank = dim**2
povm_rank = dim
state_rank = dim
num_gates = 3
num_shots = 1000

# Initial tensors

In [5]:
kraus_tensor_target, povm_psd_target, state_psd_target = get_compressed_rep_from_mgst_output(kraus_superop_target, povm_vect_target, state_vect_target, kraus_rank=kraus_rank, state_rank=state_rank, povm_rank=povm_rank)

target_psd_operators = {
    "kraus": kraus_tensor_target,
    "povm": povm_psd_target,
    "state": state_psd_target
}
print("Target operators PSD:", [op.shape for op in target_psd_operators.values()])

kraus_tensor_init_dict = get_compressed_perturbed_kraus_from_superop(superop=kraus_superop_target, rank=kraus_rank, seed=42)
kraus_tensor_init = kraus_tensor_init_dict["kraus_tensor"]

povm_psd_init, state_psd_init = get_compressed_perturbed_rep_from_mgst(povm_vect_target, state_vect_target, rank_povm=povm_rank, rank_state=state_rank)

init_operators = {
    "kraus": kraus_tensor_init,
    "povm": povm_psd_init,
    "state": state_psd_init,
}

true_superops={
    "kraus": kraus_superop_target,
    "povm": povm_vect_target,
    "state": state_vect_target
}

print(f"Init operators PSD: {[v.shape for v in init_operators.values()]}")
print(f"Target superops: {[v.shape for v in true_superops.values()]}")

Target operators PSD: [(3, 4, 2, 2), (2, 2, 2), (2, 2)]
Init operators PSD: [(3, 4, 2, 2), (2, 2, 2), (2, 2)]
Target superops: [(3, 4, 4), (2, 4), (4,)]


In [6]:
# get the MGST representation from the perturbed init operators
init_superops = get_mgst_tensors_from_psd_representation(*init_operators.values())
init_superops.keys()

dict_keys(['kraus', 'povm', 'state'])

In [7]:
seq_len_list = [1, 8, 14]
indices_dict = generate_sequence_indices(num_gates=num_gates, num_circuits=num_sequences, seq_len_list=seq_len_list)
indices_list = indices_dict["gate_indices"]
indices_list_mgst = indices_dict["gate_indices_with_negs"]

probability_matrices = compute_probability_matrices(*target_psd_operators.values(), gate_indices=indices_list, num_shots=num_shots, seed=42)
probability_matrix = probability_matrices["sampled"]

In [12]:
cost_fn_kwargs_jax = {
    "indices_list": indices_list,
    "prob_matrix": probability_matrix,
    "jit": True,
}

cost_fn_kwargs_mgst = {
    "J": indices_list_mgst,
    "y": probability_matrix,
}

# Least squares test

In [14]:
cost_jax_least_squares = cost_function_jax_mps(*init_operators.values(), **cost_fn_kwargs_jax)

at iteration 0 the new count changed to: 1
at iteration 1 the new count changed to: 2
at iteration 4 the new count changed to: 3
at iteration 10 the new count changed to: 4
at iteration 16 the new count changed to: 5
at iteration 22 the new count changed to: 6
at iteration 29 the new count changed to: 7
at iteration 36 the new count changed to: 8
at iteration 43 the new count changed to: 9
at iteration 50 the new count changed to: 10
at iteration 58 the new count changed to: 11
at iteration 66 the new count changed to: 12
at iteration 74 the new count changed to: 13
at iteration 82 the new count changed to: 14
at iteration 91 the new count changed to: 15


In [15]:
cost_jax_least_squares

Array(0.00116301, dtype=float64)

In [16]:
cost_mgst_least_squares = objf(*init_superops.values(), **cost_fn_kwargs_mgst)
cost_mgst_least_squares

0.0011630097602877863

In [17]:
jnp.allclose(cost_jax_least_squares, cost_mgst_least_squares)

Array(True, dtype=bool)

## Log-likelihood function

In [18]:
cost_jax_log_likelihood = cost_function_jax_mps(*init_operators.values(), **cost_fn_kwargs_jax, use_log_likelihood=True, num_shots=num_shots)
cost_jax_log_likelihood

at iteration 0 the new count changed to: 1
at iteration 1 the new count changed to: 2
at iteration 4 the new count changed to: 3
at iteration 10 the new count changed to: 4
at iteration 16 the new count changed to: 5
at iteration 22 the new count changed to: 6
at iteration 29 the new count changed to: 7
at iteration 36 the new count changed to: 8
at iteration 43 the new count changed to: 9
at iteration 50 the new count changed to: 10
at iteration 58 the new count changed to: 11
at iteration 66 the new count changed to: 12
at iteration 74 the new count changed to: 13
at iteration 82 the new count changed to: 14
at iteration 91 the new count changed to: 15


Array(43.58881594, dtype=float64)

In [19]:
cost_mgst_log_likelihood = objf(*init_superops.values(), **cost_fn_kwargs_mgst, mle=True)
cost_mgst_log_likelihood

43.588815940353136

# Gradient & timing

In [ ]:
cost_fn_x_least_squares = lambda x: cost_function_jax_mps(
                kraus_tensor=x, povm_psd=povm_psd_init, state_psd=state_psd_init, **cost_fn_kwargs_jax)

cost_fn_x_log_likelihood = lambda x: cost_function_jax_mps(
                kraus_tensor=x, povm_psd=povm_psd_init, state_psd=state_psd_init, **cost_fn_kwargs_jax, use_log_likelihood=True, num_shots=num_shots)

gradient_fn_least_squares = automatic_gradient(cost_fn_x_least_squares)
gradient_fn_log_likelihood = automatic_gradient(cost_fn_x_log_likelihood)

In [22]:
timing_cost_least_squares = %timeit -o cost_fn_x_least_squares(kraus_tensor_init)
timing_cost_log_likelihood = %timeit -o cost_fn_x_log_likelihood(kraus_tensor_init)

2.74 ms ± 68.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
2.73 ms ± 39.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [23]:
timing_grad_least_squares = %timeit -o gradient_fn_least_squares(kraus_tensor_init)
timing_grad_log_likelihood = %timeit -o gradient_fn_log_likelihood(kraus_tensor_init)

124 ms ± 2.24 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
127 ms ± 2.49 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
